# 第12章：异常处理（Exception Handling）

## 概念

**异常处理** = 当 LLM 调用失败时，使用备用策略保证系统稳定

```
请求 → 主处理 → 失败？ → 是 → 备用处理 → 返回结果
                ↓ 否                        ↑
              返回结果 ─────────────────────┘
```

## 异常处理策略

| 策略 | 说明 | 示例 |
|------|------|------|
| **try/except** | 捕获异常，返回兜底结果 | API 超时返回默认回答 |
| **Fallback** | 主方案失败，切换备用方案 | 主模型失败换备用模型 |
| **Retry** | 失败后重试，可设置次数 | 网络波动自动重试 |

## 代码演示

使用 LangChain 实现三种异常处理策略

In [2]:
import os
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
load_dotenv()

# 主模型
llm_primary = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

# 备用模型（模拟，使用相同模型但不同参数）
llm_fallback = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0.7
)

print("模型初始化完成！")

模型初始化完成！


## 策略1：try/except 捕获异常

捕获 LLM 调用异常，返回兜底结果

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。请简洁回答。"),
    ("user", "{question}")
])

chain = prompt | llm_primary | StrOutputParser()

def safe_call(question: str, default: str = "抱歉，暂时无法回答。") -> str:
    """带 try/except 的安全调用"""
    try:
        return chain.invoke({"question": question})
    except Exception as e:
        print(f"调用失败: {e}")
        return default

# 测试
print("=== 策略1：try/except ===")
result = safe_call("什么是AI？")
print(f"正常调用: {result[:50]}...")

# 模拟失败场景
result = safe_call("", default="请输入有效问题。")
print(f"空问题: {result}")

=== 策略1：try/except ===
正常调用: AI（人工智能）是计算机科学的一个分支，旨在创建能够模拟人类智能的系统，包括学习、推理、感知和决策等...
空问题: It looks like your message is empty! How can I help you today? 😊


## 策略2：Fallback 备用方案

主方案失败时，自动切换备用方案

In [5]:
# 主链
primary_chain = prompt | llm_primary | StrOutputParser()

# 备用链（不同提示词）
fallback_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。请用最简洁的语言回答。"),
    ("user", "{question}")
])
fallback_chain = fallback_prompt | llm_fallback | StrOutputParser()

def call_with_fallback(question: str) -> str:
    """主方案失败时切换备用方案"""
    try:
        result = primary_chain.invoke({"question": question})
        print("[使用主方案]")
        return result
    except Exception as e:
        print(f"[主方案失败: {e}，切换备用方案]")
        try:
            result = fallback_chain.invoke({"question": question})
            return result
        except Exception as e2:
            print(f"[备用方案也失败: {e2}]")
            return "所有方案均失败，请稍后重试。"

# 测试
print("=== 策略2：Fallback ===")
result = call_with_fallback("用一句话解释量子计算")
print(f"结果: {result}")

=== 策略2：Fallback ===
[使用主方案]
结果: 量子计算利用量子比特的叠加和纠缠特性，使计算机能同时探索巨量可能性，从而在特定问题上远超传统计算机的处理速度。


## 策略3：Retry 重试机制

失败后自动重试，支持指数退避

In [6]:
def call_with_retry(question: str, max_retries: int = 3, base_delay: float = 1.0) -> str:
    """带重试的调用，支持指数退避"""
    for attempt in range(max_retries):
        try:
            result = chain.invoke({"question": question})
            print(f"[第{attempt + 1}次尝试成功]")
            return result
        except Exception as e:
            delay = base_delay * (2 ** attempt)  # 指数退避
            print(f"[第{attempt + 1}次尝试失败: {e}，{delay}秒后重试]")
            time.sleep(delay)
    
    return f"{max_retries}次重试均失败。"

# 测试
print("=== 策略3：Retry ===")
result = call_with_retry("什么是机器学习？")
print(f"结果: {result[:50]}...")

=== 策略3：Retry ===
[第1次尝试成功]
结果: 机器学习是人工智能的一个分支，它使计算机系统能够通过**数据**和**经验**自动学习和改进，而无需...


## 完整的异常处理链

组合三种策略：try/except → Fallback → Retry

In [7]:
def robust_call(question: str) -> str:
    """完整的异常处理链"""
    # 第1层：Retry
    for attempt in range(2):
        try:
            # 第2层：Fallback
            try:
                result = primary_chain.invoke({"question": question})
                return f"[主方案] {result}"
            except Exception:
                result = fallback_chain.invoke({"question": question})
                return f"[备用方案] {result}"
        except Exception as e:
            if attempt < 1:
                print(f"重试中... ({attempt + 1}/2)")
                time.sleep(1)
    
    # 第3层：兜底
    return "[兜底] 抱歉，暂时无法回答，请稍后重试。"

# 测试
print("=== 完整异常处理链 ===")
result = robust_call("深度学习和机器学习的区别")
print(f"结果: {result[:60]}...")

=== 完整异常处理链 ===
结果: [主方案] # 深度学习与机器学习的区别

## 关系总览

```
人工智能 ⊃ 机器学习 ⊃ 深度学习
```

深...


## 异常处理流程

```
┌─────────────────────────────────────────────────────────┐
│                 异常处理流程                              │
├─────────────────────────────────────────────────────────┤
│  请求                                                   │
│    ↓                                                    │
│  主方案调用 ──成功──→ 返回结果                            │
│    │                                                    │
│    失败                                                 │
│    ↓                                                    │
│  备用方案调用 ──成功──→ 返回结果                          │
│    │                                                    │
│    失败                                                 │
│    ↓                                                    │
│  重试（最多N次）──成功──→ 返回结果                        │
│    │                                                    │
│    全部失败                                              │
│    ↓                                                    │
│  返回兜底结果                                            │
└─────────────────────────────────────────────────────────┘
```

## 三种策略对比

| 策略 | 适用场景 | 优点 | 缺点 |
|------|----------|------|------|
| **try/except** | 任何调用 | 简单、防止崩溃 | 静默失败 |
| **Fallback** | 有备用方案 | 保证可用性 | 增加复杂度 |
| **Retry** | 临时故障 | 自动恢复 | 增加延迟 |

## 实际应用场景

- **API 超时**：Retry + 指数退避
- **模型不可用**：Fallback 到备用模型
- **格式错误**：重试 + 修正提示词
- **速率限制**：Retry + 延迟